[![Buy Me A Coffee](https://img.shields.io/badge/Buy%20Me%20A%20Coffee-support%20my%20work-FFDD00?style=flat&labelColor=101010&logo=buy-me-a-coffee&logoColor=white)](https://www.buymeacoffee.com/r0mymendez)

---

# From Coffee Products to AI Search: Building a Serverless Semantic Search Architecture with Amazon S3 Vectors and Bedrock

![img](img/1-preview.png)

In recent months, we have increasingly incorporated artificial intelligence into our solutions, and with it a recurring need has emerged: searching and querying our own data using natural language efficiently.

Use cases such as semantic search or building solutions based on Retrieval-Augmented Generation (RAG) are no longer optional. Today, we need to understand the meaning of text, combine it with structured filters, and do so in an efficient and scalable way.
In this article, I explore a recent alternative within the AWS ecosystem: Amazon S3 Vectors 🪣, a serverless approach for vector storage and querying that aims to balance scalability, simplicity, and cost.

To make it more concrete (and a bit more entertaining)...we will work with a dataset of coffee products ☕ and build a complete flow that goes from generating embeddings with Amazon Bedrock 🧠 to an application deployed on AWS with Streamlit ✨, which allows natural language searches combined with filters.


---

# What is Amazon S3 Vectors?
**Amazon S3 Vectors** is a new type of storage within Amazon S3 designed specifically to natively **store and query vectors**.
 In addition to storing vectors, this type of bucket allows associating **structured metadata**, which enables queries that combine **semantic search** with filters on those attributes.
Vector buckets support searches based on distance metrics, such as:
* **Cosine similarity**: measures how similar two vectors are based on the angle between them, and is very common in text embeddings.
* **Euclidean distance**: measures the “geometric” distance between two vectors in space.
Unlike traditional vector databases, Amazon S3 Vectors makes it possible to **implement a fully serverless architecture**, achieving a good balance between `scalability`, `operational` `simplicity`, and `cost`.
Below are some of the main benefits of using this functionality:

![img2](img/2-s3-vectors.png)

## How do vectors work in Amazon S3?
Amazon S3 Vectors is based on the following main components:

**🪣 1. Vector buckets**
These are specialized buckets optimized for vector storage.
They support encryption and organize data internally through **vector indexes**, which enables efficient large-scale searches.

**🧭 2. Vector indexes**
An index defines how vectors are stored and queried within the bucket.
In addition to the vector, it allows associating **metadata**, which can later be used in queries through filters with a syntax similar to well-known operators, such as those used in MongoDB.

**🔍 3. Queries**
Queries are based on **similarity searches**, using the distance metric configured when creating the index, such as **cosine** or **Euclidean**.
These searches can be combined with metadata filters to refine results and reduce ambiguities.

**⚙️ 4. API**
**Amazon S3 Vectors** exposes an API that allows querying data through operations such as `QueryVectors`.
These queries can be executed using tools like the **AWS CLI** or **Boto3**, combining a query vector with metadata-based filters and parameters such as the number of results to return or whether to include the distance between vectors.

---

# Process Flow
The previous image shows the complete workflow to implement semantic search using Amazon S3 Vectors, divided into three main stages:

![img-3](img/3-process-flow.png)

## 1️⃣ Generate Vector Embeddings
The process starts from the input documents. These documents are sent to an embeddings model, in this case **AWS Titan** through **Amazon Bedrock**, which transforms the text into numerical vectors.
At this stage, not only are the vectors generated, but metadata describing each document is also associated.

---

## 2️⃣ Store Vector Data
The generated vectors, together with their metadata, are stored in an **S3 Vector Bucket**.
Within the bucket, the data is organized through one or more **vector indexes**, defined with a specific distance metric.
Being integrated into AWS, this data can be consumed by other services such as **Amazon Bedrock**, **Amazon SageMaker**, or **Amazon OpenSearch**.

---

## 3️⃣ Semantic Search via Vector Index
To perform a search, a natural language query is transformed again into a vector using the same embeddings model.
This query vector, together with metadata filters and the topK parameter, is used to query the vector index and retrieve the most semantically similar results.



In [9]:
# import necessary libraries

import boto3,json,os
from dotenv import load_dotenv
import pandas as pd
from datetime import datetime
from tqdm import tqdm

In [10]:
class EmbeddingsGenerator:
    """Class to generate embeddings using Amazon Bedrock Titan Embedding Model"""
    def __init__(self, 
                 MODEL_NAME:str='amazon.titan-embed-text-v2:0', 
                 AWS_ACCESS_KEY_ID:str='', 
                 AWS_SECRET_ACCESS_KEY:str='',
                 AWS_REGION:str=''
                 ):
        self.MODEL_NAME = MODEL_NAME
        self.AWS_ACCESS_KEY_ID = AWS_ACCESS_KEY_ID
        self.AWS_SECRET_ACCESS_KEY = AWS_SECRET_ACCESS_KEY
        self.AWS_REGION = AWS_REGION

    def create_client(self):
        """create boto3 client for Bedrock Runtime"""
        client = boto3.client(
                service_name='bedrock-runtime',
                region_name=self.AWS_REGION,
                aws_access_key_id=self.AWS_ACCESS_KEY_ID,
                aws_secret_access_key=self.AWS_SECRET_ACCESS_KEY
            )
        return client
    
    def get_embeddings(self, text:str):
        """Generate embeddings for a given text"""
        client = self.create_client()
        response = client.invoke_model(
            modelId=self.MODEL_NAME,
            body=json.dumps({
                "inputText": text
            })
        )
        response_body = json.loads(response['body'].read())
        embeddings = response_body['embedding']
        return embeddings
    
    def generate_embeddings_batch(self, texts:list):
        """Generate embeddings for a batch of texts"""
        embeddings_list = []
        for text in tqdm(texts):
            embeddings = self.get_embeddings(text)
            embeddings_list.append(embeddings)
        return embeddings_list



In [26]:
class S3:
    """Class to handle S3 operations including uploading files and vector data"""
    def __init__(self, 
                 AWS_ACCESS_KEY_ID:str='', 
                 AWS_SECRET_ACCESS_KEY:str='',
                 AWS_REGION:str='',
                 AWS_BUCKET_NAME:str='',
                 AWS_BUCKET_VECTOR_NAME:str='',
                 AWS_INDEX_VECTOR_NAME:str=''
                 ):
        self.AWS_ACCESS_KEY_ID = AWS_ACCESS_KEY_ID
        self.AWS_SECRET_ACCESS_KEY = AWS_SECRET_ACCESS_KEY
        self.AWS_REGION = AWS_REGION
        self.AWS_BUCKET_NAME = AWS_BUCKET_NAME
        self.AWS_BUCKET_VECTOR_NAME = AWS_BUCKET_VECTOR_NAME
        self.AWS_INDEX_VECTOR_NAME = AWS_INDEX_VECTOR_NAME

    def create_client(self, service_name:str='s3'):
        """
        Create a boto3 client for the specified AWS service.
        """
        s3_client = boto3.client(
            service_name=service_name,
            region_name=self.AWS_REGION,
            aws_access_key_id=self.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=self.AWS_SECRET_ACCESS_KEY
        )
        return s3_client

    def upload_file(self, file_name:str, object_name:str):
        """
        Upload a file to an S3 bucket.
        """
        s3_client = self.create_client()
        s3_client.upload_file(Filename=file_name, Bucket=self.AWS_BUCKET_NAME, Key=object_name)
        print(f"File {file_name} uploaded to bucket {self.AWS_BUCKET_NAME} as {object_name}")

    def upload_vector_data(self, data:list, batch_size:int=100):
        """
        Upload vector data to S3 Vectors in batches with tqdm for progress tracking.
        batchsize: it is the number of vectors per batch to avoid exceeding maximum size.
        """
        s3_vector_client = self.create_client(service_name='s3vectors')

        # Helper for chunking data into batches
        def chunked(lst, size):
            for i in range(0, len(lst), size):
                yield lst[i:i + size]

        batches = list(chunked(data, batch_size))

        # see the progress of the upload
        for i, batch in enumerate(tqdm(batches, desc="Uploading batches"), start=1):
            try:
                s3_vector_client.put_vectors(
                    vectorBucketName=self.AWS_BUCKET_VECTOR_NAME,
                    indexName=self.AWS_INDEX_VECTOR_NAME,
                    vectors=batch
                )
            except Exception as e:
                print(f"Error uploading batch {i}: {e}")
    
    def query_embedding(self, 
               query_embedding:list, 
               filter_data:dict=None,
                top_k=3):
        """Perform complete search with text and filters"""
        s3_vector_client = self.create_client(service_name='s3vectors')
        
        # Prepare base parameters
        query_params = {
            "vectorBucketName": self.AWS_BUCKET_VECTOR_NAME,
            "indexName": self.AWS_INDEX_VECTOR_NAME,
            "queryVector": {"float32": query_embedding},
            "topK": top_k,
            "returnDistance": True,
            "returnMetadata": True
        }
        
        # Only add filter if exists
        if filter_data:
            query_params["filter"] = filter_data
        
        # Execute search
        query_result = s3_vector_client.query_vectors(**query_params)
        return query_result['vectors']


### Dataset Preparation and Text Construction for Embeddings
_Filtering coffee brands and creating the input text for semantic indexing_

In [ ]:
data_coffee = pd.read_parquet('data/raw/dataset_clean_coffee.parquet', engine='fastparquet')

# Filter coffee brands
_coffee_brands = ['starbucks', 'folgers','green mountain coffee roasters','nespresso','nescafé',
 'keurig','maxwell house','lavazza','tassimo','illy','dolce gusto','mccafé',"dunkin' donuts",'fresh roasted coffee']
data_coffee_filter = data_coffee[data_coffee['shop_name'].isin(_coffee_brands)].reset_index(drop=True)
data_coffee_filter = data_coffee_filter[~data_coffee_filter['price'].isna()].reset_index(drop=True)

# Create the columna with data to embebed vectors
data_coffee_filter['full_text'] = [ f"""title: {data_coffee_filter['title'][item]} \n description: {data_coffee_filter['full_description'][item]} \n category: {data_coffee_filter['category'][item]} \n"""
                                   for item in range(data_coffee_filter.shape[0])]

### Generating Embeddings with Amazon Titan
_Using Amazon Bedrock to convert product text into vector representations_

In [ ]:
# Load environment variables
load_dotenv()
AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY')
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')
AWS_REGION = os.getenv('AWS_REGION')
AWS_BUCKET_NAME = "coffee-products-tutorial-full-data"
AWS_BUCKET_VECTOR_NAME  = "coffee-products-tutorial"
AWS_INDEX_VECTOR_NAME ="idx-coffee-products"


#  EmbeddingsGenerator
emb_generator = EmbeddingsGenerator(
    AWS_ACCESS_KEY_ID=AWS_ACCESS_KEY_ID,
    AWS_SECRET_ACCESS_KEY=AWS_SECRET_ACCESS_KEY,
    AWS_REGION=AWS_REGION
)

full_text_list = data_coffee_filter['full_text'].to_list()
full_text_embeddings = emb_generator.generate_embeddings_batch(full_text_list)

# Add embeddings to dataframe
data_coffee_filter['embeddings'] = full_text_embeddings

In [35]:
vector_data = []

for i in range(data_coffee_filter.shape[0]):
    vector_data.append({
        "key": str(data_coffee_filter['id'][i]),  # always need to be string
        "data": {
            "float32": data_coffee_filter['embeddings'][i]
        },
        "metadata": {
            "average": float(data_coffee_filter['average_rating'][i]),
            "rating_number": int(data_coffee_filter['rating_number'][i]),
            "price": float(data_coffee_filter['price'][i]),
            "shop_name": str(data_coffee_filter['shop_name'][i])
        }
    })


### Store Data in Amazon S3 and S3 Vectors
_Uploading processed datasets and persisting vector embeddings with metadata_

In [ ]:
s3 = S3(
    AWS_ACCESS_KEY_ID=AWS_ACCESS_KEY_ID,
    AWS_SECRET_ACCESS_KEY=AWS_SECRET_ACCESS_KEY,
    AWS_REGION=AWS_REGION,
    AWS_BUCKET_NAME=AWS_BUCKET_NAME,
    AWS_BUCKET_VECTOR_NAME=AWS_BUCKET_VECTOR_NAME,
    AWS_INDEX_VECTOR_NAME=AWS_INDEX_VECTOR_NAME
)


today = datetime.now().strftime("%Y-%m-%d")

# Upload files to S3 in raw folder
s3.upload_file(
    file_name='data/dataset_clean_coffee.parquet',
    object_name=f'raw/{today}_dataset_coffee.parquet'
)

# Upload  file to S3 with embeddings in stage folder
s3.upload_file(
    file_name='data/dataset_clean_coffee_embeddings.parquet',
    object_name=f'stage/{today}_dataset_coffee_embeddings.parquet'
)

s3.upload_vector_data(vector_data)

Uploading batches: 100%|██████████| 12/12 [00:11<00:00,  1.00it/s]


In [ ]:
# Do similarity search with metadata filtering
input_text = "instant coffee sweet creamy vanilla flavor"
query_embedding = emb_generator.get_embeddings(text=input_text)

# Get results
s3.query_embedding( query_embedding=query_embedding)

/Users/romina.mendez/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


[{'distance': 0.41610199213027954,
  'key': 'de46725d-ef52-47ca-80e2-f1ba82c0353d',
  'metadata': {'average': 4.4,
   'rating_number': 248,
   'shop_name': 'nescafé',
   'price': 11.48}},
 {'distance': 0.457456111907959,
  'key': 'afbeefeb-946e-4f0a-88e1-3883797ad4ae',
  'metadata': {'price': 44.99,
   'rating_number': 18250,
   'shop_name': 'maxwell house',
   'average': 4.6}},
 {'distance': 0.46814537048339844,
  'key': '36832fed-a104-4485-8bb9-6f4252d62c77',
  'metadata': {'rating_number': 2,
   'shop_name': 'maxwell house',
   'price': 16.97,
   'average': 4.0}}]

In [ ]:
# Do similarity search with metadata filtering with shop_name nescafé
s3.query_embedding( query_embedding=query_embedding, 
                   filter_data={"shop_name": "nescafé"})

[{'distance': 0.41610199213027954,
  'key': 'de46725d-ef52-47ca-80e2-f1ba82c0353d',
  'metadata': {'price': 11.48,
   'shop_name': 'nescafé',
   'average': 4.4,
   'rating_number': 248}},
 {'distance': 0.47703248262405396,
  'key': '03915b9f-e592-40ec-b806-bd06b4213d90',
  'metadata': {'price': 13.4,
   'average': 3.6,
   'shop_name': 'nescafé',
   'rating_number': 471}},
 {'distance': 0.514411211013794,
  'key': '5037ea28-b789-427a-9b1f-d825ad68dd2d',
  'metadata': {'rating_number': 3052,
   'shop_name': 'nescafé',
   'average': 4.4,
   'price': 17.75}}]

In [30]:
s3.query_embedding( query_embedding=query_embedding, 
                   filter_data={"average": {"$gte": 4.2}})

[{'distance': 0.41610199213027954,
  'key': 'de46725d-ef52-47ca-80e2-f1ba82c0353d',
  'metadata': {'price': 11.48,
   'average': 4.4,
   'shop_name': 'nescafé',
   'rating_number': 248}},
 {'distance': 0.457456111907959,
  'key': 'afbeefeb-946e-4f0a-88e1-3883797ad4ae',
  'metadata': {'rating_number': 18250,
   'shop_name': 'maxwell house',
   'average': 4.6,
   'price': 44.99}},
 {'distance': 0.48257017135620117,
  'key': 'a61c87db-3b3b-47eb-8526-f510548667ab',
  'metadata': {'shop_name': 'maxwell house',
   'price': 33.21,
   'average': 4.5,
   'rating_number': 812}}]

In [31]:
s3.query_embedding( query_embedding=query_embedding, 
                   filter_data={
        "$and": [
            {"average": {"$gte": 4.2}},
            {"price": {"$lte": 20.0}}
        ]
    })

[{'distance': 0.41610199213027954,
  'key': 'de46725d-ef52-47ca-80e2-f1ba82c0353d',
  'metadata': {'rating_number': 248,
   'average': 4.4,
   'price': 11.48,
   'shop_name': 'nescafé'}},
 {'distance': 0.5135153532028198,
  'key': '7d702dc6-b45a-4141-b968-378aca6c8215',
  'metadata': {'price': 18.98,
   'rating_number': 196,
   'shop_name': 'illy',
   'average': 4.5}},
 {'distance': 0.5135153532028198,
  'key': '2a29fb82-45d2-46dc-a473-274face4eed7',
  'metadata': {'average': 4.5,
   'price': 18.7,
   'shop_name': 'illy',
   'rating_number': 202}}]